<a href="https://colab.research.google.com/github/gulshan0201/DL/blob/main/DL_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
pip install torch

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# Training sentences
sentences = [
    "i love you",
    "i love machine learning",
    "i love deep learning",
    "i love natural language processing"
]

# Tokenization
words = set()
for s in sentences:
    for w in s.split():
        words.add(w)

word2idx = {w:i for i,w in enumerate(words)}
idx2word = {i:w for w,i in word2idx.items()}

vocab_size = len(word2idx)

# Convert sentences to numbers
def encode(sentence):
    return [word2idx[w] for w in sentence.split()]

data = [encode(s) for s in sentences]

max_len = max(len(s) for s in data)

# Padding
for i in range(len(data)):
    data[i] += [0]*(max_len-len(data[i]))

data = torch.tensor(data)

# Autoencoder Model
class AutoEncoder(nn.Module):

    def __init__(self,vocab_size,embed_size=16,hidden=32):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size,embed_size)
        self.encoder = nn.LSTM(embed_size,hidden,batch_first=True)
        self.decoder = nn.LSTM(embed_size,hidden,batch_first=True)
        self.fc = nn.Linear(hidden,vocab_size)

    def forward(self,x):

        x = self.embedding(x)

        _,(h,c) = self.encoder(x)

        out,_ = self.decoder(x,(h,c))

        out = self.fc(out)

        return out

model = AutoEncoder(vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.01)

# Training
for epoch in range(200):

    optimizer.zero_grad()

    output = model(data)

    loss = criterion(output.view(-1,vocab_size),data.view(-1))

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print("epoch:",epoch,"loss:",loss.item())

# Prediction
test = "i love"
test_ids = [word2idx[w] for w in test.split()]
test_ids += [0]*(max_len-len(test_ids))

test_tensor = torch.tensor([test_ids])

pred = model(test_tensor)

pred_word = torch.argmax(pred[0][2]).item()

print("Predicted word:",idx2word[pred_word])

epoch: 0 loss: 2.2419724464416504
epoch: 20 loss: 0.1687498241662979
epoch: 40 loss: 0.013623815961182117
epoch: 60 loss: 0.005405276082456112
epoch: 80 loss: 0.0036238315515220165
epoch: 100 loss: 0.0027803690172731876
epoch: 120 loss: 0.00223649968393147
epoch: 140 loss: 0.0018415559316053987
epoch: 160 loss: 0.0015409684274345636
epoch: 180 loss: 0.00131313968449831
Predicted word: deep


In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

# Device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training sentences
sentences = [
    "i love you",
    "i love machine learning",
    "i love deep learning",
    "i love natural language processing"
]

# Build vocabulary
PAD = "<PAD>"
words = {PAD}

for s in sentences:
    words.update(s.split())

word2idx = {w:i for i,w in enumerate(sorted(words))}
idx2word = {i:w for w,i in word2idx.items()}

vocab_size = len(word2idx)
pad_idx = word2idx[PAD]

# Encode sentences
def encode(sentence):
    return [word2idx[w] for w in sentence.split()]

encoded = [encode(s) for s in sentences]
max_len = max(len(s) for s in encoded)

# Dataset
class TextDataset(Dataset):
    def __init__(self,data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        x = self.data[idx]
        x = x + [pad_idx]*(max_len-len(x))
        return torch.tensor(x)

dataset = TextDataset(encoded)
loader = DataLoader(dataset,batch_size=2,shuffle=True)

# Model
class AutoEncoder(nn.Module):
    def __init__(self,vocab,embed=32,hidden=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab,embed,padding_idx=pad_idx)
        self.encoder = nn.LSTM(embed,hidden,batch_first=True)
        self.decoder = nn.LSTM(embed,hidden,batch_first=True)
        self.fc = nn.Linear(hidden,vocab)

    def forward(self,x):

        emb = self.embedding(x)

        _,(h,c) = self.encoder(emb)

        dec_out,_ = self.decoder(emb,(h,c))

        out = self.fc(dec_out)

        return out

model = AutoEncoder(vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = torch.optim.Adam(model.parameters(),lr=0.003)

# Training
epochs = 150

for epoch in range(epochs):

    total_loss = 0

    for batch in loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        output = model(batch)

        loss = criterion(output.view(-1,vocab_size),batch.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch} Loss {total_loss:.4f}")

# Prediction
def predict(text):

    tokens = text.split()

    ids = [word2idx[w] for w in tokens]
    ids += [pad_idx]*(max_len-len(ids))

    x = torch.tensor([ids]).to(device)

    with torch.no_grad():
        out = model(x)

    next_pos = len(tokens)
    pred_id = torch.argmax(out[0][next_pos]).item()

    return idx2word[pred_id]

# Test
print("Input: i love")
print("Prediction:",predict("i love"))

Epoch 0 Loss 4.4927
Epoch 20 Loss 0.2885
Epoch 40 Loss 0.0307
Epoch 60 Loss 0.0146
Epoch 80 Loss 0.0093
Epoch 100 Loss 0.0067
Epoch 120 Loss 0.0050
Epoch 140 Loss 0.0039
Input: i love
Prediction: natural
